In [1]:
import os
import numpy as np
import nibabel as nib
from neuromaps import datasets, resampling, stats, images, nulls

In [2]:
# Load fooof data
project_root = r'C:\Users\Cristina\megtusalen'
apexp_file = os.path.join(project_root,'data','sources','apexp','FAM-001_rECpost_MEGMAG_apexp_clean.nii')

# Receptor map
dubois_file = datasets.fetch_annotation(source='dubois2015', return_single=True)


[References] Please cite the following papers if you are using this data:

  For {'source': 'dubois2015', 'desc': 'abp688', 'space': 'MNI152', 'res': '1mm'}:
  [primary]:
    Jonathan M DuBois, Olivier G Rousset, Jared Rowley, Manuel Porras-Betancourt, Andrew J Reader, Aurelie Labbe, Gassan Massarweh, Jean-Paul Soucy, Pedro Rosa-Neto, and Eliane Kobayashi. Characterization of age/sex and the regional distribution of mglur5 availability in the healthy human brain measured by high-resolution [11 c] abp688 pet. European journal of nuclear medicine and molecular imaging, 43(1):152–162, 2016.
    Justine Y Hansen, Golia Shafiei, Ross D Markello, Kelly Smart, Sylvia ML Cox, Martin Nørgaard, Vincent Beliveau, Yanjun Wu, Jean-Dominique Gallezot, Étienne Aumont, and others. Mapping neurotransmitter systems to the structural and functional organization of the human neocortex. Nature neuroscience, 25(11):1569–1581, 2022.
  [secondary]:
    


In [3]:
# Resample dubois to the apexp grid (lower resolution)
dubois, apexp = resampling.resample_images(src=dubois_file, trg=apexp_file,
                                            src_space='MNI152',
                                            trg_space='MNI152',
                                            resampling='transform_to_trg')


C:\Users\Cristina\miniconda3\envs\pysurfer_env\lib\site-packages\neuromaps\transforms.py:254: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  out = nimage.resample_to_img(load_nifti(img), load_nifti(target),
C:\Users\Cristina\miniconda3\envs\pysurfer_env\lib\site-packages\nilearn\image\resampling.py:805: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  return resample_img(


In [4]:
# Correlate dubois and apexp
corr = stats.compare_images(dubois, apexp)
print(f'Correlation: r = {corr:.04f}')

Correlation: r = 0.0299


In [5]:
 # Load fake parcellation (brain mask)
parcellation_file = os.path.join(project_root,'data','sources','apexp','parcellation.nii')
parcellation_data = nib.load(parcellation_file)

# Get binary mask from all the values in parcellation_data that are non-zero
brain_mask = parcellation_data.get_fdata() != 0
brain_mask_flat = brain_mask.flatten()

# Extract dubois data in the brain mask
dubois_data = images.load_data(dubois)
dubois_data_flat = dubois_data.flatten()
dubois_data_flat = dubois_data_flat[brain_mask_flat]

In [9]:
# Compute the spatial nulls
n_perm = 100
rotated = nulls.burt2020(dubois_data_flat, atlas = 'mni152', density = '2mm', parcellation=parcellation_data, n_perm=n_perm, seed=1234)

In [11]:
# Embed spatial nulls into the original 3d grid
nulls_4d = np.zeros(brain_mask.shape + (n_perm,), dtype=rotated.dtype)
nulls_4d.reshape(-1, n_perm)[brain_mask_flat, :] = rotated

In [12]:
# Compute correlation with pvalue
corr, pval = stats.compare_images(dubois, apexp, nulls=nulls_4d)
print(f'Correlation: r = {corr:.04f}, p = {pval:.04f}')

Correlation: r = 0.0299, p = 0.8020
